<a href="https://colab.research.google.com/github/Sanchezx2/TraductorGoogleColab/blob/main/Practica7CarlosSanchez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Práctica 7: Traducción Avanzada e Interactiva en Google Colab
En este cuaderno exploraremos la librería `googletrans` para la traducción de textos empleando la infraestructura de Google.

Además de los ejemplos básicos propuestos, hemos añadido las siguientes **mejoras y ampliaciones**:
1. **Traducción a múltiples idiomas** (Alemán, Francés, etc.).
2. **Detección automática** del idioma de origen.
3. **Interfaz gráfica interactiva (GUI)** integrada en Colab utilizando `ipywidgets` para que cualquier usuario pueda traducir sin tocar el código.

In [ ]:
# Instalamos la versión específica para evitar el error de "NoneType" de versiones anteriores
!pip install googletrans==4.0.0-rc1

# Importamos las librerías necesarias para la traducción y la interfaz gráfica
from googletrans import Translator, LANGUAGES
import ipywidgets as widgets
from IPython.display import display, clear_output

# Instanciamos el traductor
translator = Translator()
print("¡Librerías instaladas y listas para usar!")

¡Librerías instaladas y listas para usar!


## 1. Ejemplos Básicos del Guion
Traducciones directas desde el código, indicando el idioma de origen (`src`) y el de destino (`dest`).

In [ ]:
# Ejemplo 1: Inglés a Español
resultado = translator.translate('Hello, how are you?', src='en', dest='es')
print(f"EN -> ES: {resultado.text}")

# Ejemplo 2: Inglés a Español
texto = 'This is a powerful translation tool!'
resultado2 = translator.translate(texto, src='en', dest='es')
print(f"EN -> ES: {resultado2.text}")

# Ampliación: Español a Alemán ('de') y Español a Francés ('fr')
texto_es = 'La computación distribuida nos permite usar el poder de la nube.'

resultado_de = translator.translate(texto_es, src='es', dest='de')
print(f"ES -> DE (Alemán): {resultado_de.text}")

resultado_fr = translator.translate(texto_es, src='es', dest='fr')
print(f"ES -> FR (Francés): {resultado_fr.text}")

EN -> ES: ¿Hola, cómo estás?
EN -> ES: ¡Esta es una poderosa herramienta de traducción!
ES -> DE (Alemán): Durch verteiltes Computing können wir die Leistungsfähigkeit der Cloud nutzen.
ES -> FR (Francés): L'informatique distribuée nous permet d'utiliser la puissance du cloud.


## 2. Ampliación: Detección Automática de Idioma
La librería permite detectar en qué idioma está escrito un texto sin decírselo previamente, usando el método `detect()`.

In [ ]:
textos_misteriosos = [
    "Buongiorno a tutti",
    "Ich lerne gerne Python",
    "Això és fantàstic"
]

for t in textos_misteriosos:
    deteccion = translator.detect(t)

    # Extraemos el nombre del idioma
    idioma = LANGUAGES.get(deteccion.lang, deteccion.lang).capitalize()

    # SOLUCIÓN: Comprobamos si Google nos ha dado un valor de confianza o si está vacío (None)
    if deteccion.confidence is not None:
        confianza_texto = f"{deteccion.confidence * 100:.1f}%"
    else:
        confianza_texto = "No disponible"

    print(f"Texto: '{t}' | Idioma detectado: {idioma} (Confianza: {confianza_texto})")

Texto: 'Buongiorno a tutti' | Idioma detectado: Italian (Confianza: No disponible)
Texto: 'Ich lerne gerne Python' | Idioma detectado: German (Confianza: No disponible)
Texto: 'Això és fantàstic' | Idioma detectado: Catalan (Confianza: No disponible)


## 3. Ampliación Estrella: Traductor Interactivo
Para cumplir con el requisito de crear un **cuaderno interactivo**, a continuación se despliega una pequeña interfaz de usuario. Escribe tu texto, selecciona el idioma y pulsa el botón.

In [26]:
import ipywidgets as widgets
from IPython.display import display
from googletrans import Translator, LANGUAGES

translator = Translator()

# 1. Crear los elementos de la interfaz
opciones_idioma = {'Español': 'es', 'Inglés': 'en', 'Francés': 'fr', 'Alemán': 'de', 'Italiano': 'it', 'Portugués': 'pt', 'Japonés': 'ja'}
desplegable_idioma = widgets.Dropdown(
    options=opciones_idioma,
    value='en', # Por defecto traduce al inglés
    description='Traducir a:',
)

etiqueta_origen = widgets.Label(value='Idioma detectado: (Escribe y pulsa ESPACIO para traducir)')
etiqueta_destino = widgets.Label(value='Traducción:')

texto_entrada = widgets.Textarea(
    placeholder='Escribe el texto aquí y pulsa ESPACIO al terminar una palabra...',
    layout=widgets.Layout(width='95%', height='150px')
)

texto_salida = widgets.Textarea(
    placeholder='La traducción aparecerá aquí...',
    disabled=True, # El usuario no puede escribir aquí
    layout=widgets.Layout(width='95%', height='150px')
)

# 2. Función de traducción directa (sin temporizadores problemáticos)
def realizar_traduccion(texto, destino):
    if not texto.strip():
        texto_salida.value = ""
        etiqueta_origen.value = 'Idioma detectado: (Escribe y pulsa ESPACIO para traducir)'
        return

    try:
        resultado = translator.translate(texto, dest=destino)
        idioma_detectado = LANGUAGES.get(resultado.src, resultado.src).capitalize()
        etiqueta_origen.value = f'Idioma detectado: {idioma_detectado}'
        texto_salida.value = resultado.text
    except Exception as e:
        texto_salida.value = "Error de conexión. Intenta borrar una letra y volver a escribirla."

# 3. Función inteligente que detecta cuándo terminas de escribir una palabra
def al_cambiar_texto(change):
    texto = change.new
    # Traduce SOLO si el texto termina en espacio, intro o puntuación (o si lo borras todo)
    if not texto or texto[-1] in [' ', '\n', '.', ',', '!', '?']:
        realizar_traduccion(texto, desplegable_idioma.value)

def al_cambiar_idioma(change):
    # Traduce de golpe si cambias el idioma en el desplegable
    realizar_traduccion(texto_entrada.value, change.new)

# 4. Conectar los eventos a las funciones
texto_entrada.observe(al_cambiar_texto, names='value')
desplegable_idioma.observe(al_cambiar_idioma, names='value')

# 5. Diseñar la vista en dos columnas
columna_izquierda = widgets.VBox([etiqueta_origen, texto_entrada], layout=widgets.Layout(width='50%'))
columna_derecha = widgets.VBox([etiqueta_destino, texto_salida], layout=widgets.Layout(width='50%'))
pantallas_juntas = widgets.HBox([columna_izquierda, columna_derecha])

# 6. Mostrar todo (Título GIGANTE y limpieza de código)

url_de_mi_foto = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcT3C-niuK8_f9sveR14w9lNQiAlW5cE-v5IzQ&s"
titulo_con_icono = f"""
<div style="display: flex; align-items: center; margin-bottom: 30px;">
    <img src="{url_de_mi_foto}" style="width: 200px; height: 200px; border-radius: 8px; object-fit: cover; margin-right: 30px;">
    <div>
        <h1 style="margin: 0; font-size: 4em; color: #ff5722; font-weight: 900; line-height: 1.1; text-transform: uppercase;">
            🔥 THE ULTIMATE<br>MEGA-TRADUCTOR<br>3000 PRO MAX 🔥
        </h1>
        <p style="margin: 15px 0 0 5px; font-size: 1.4em; font-style: italic; color: #757575; font-weight: bold;">
            "TraductorKirkificado"
        </p>
    </div>
</div>
"""

display(widgets.HTML(titulo_con_icono))
display(desplegable_idioma)
display(pantallas_juntas)

HTML(value='\n<div style="display: flex; align-items: center; margin-bottom: 30px;">\n    <img src="https://en…

Dropdown(description='Traducir a:', index=1, options={'Español': 'es', 'Inglés': 'en', 'Francés': 'fr', 'Alemá…